# 01 — Data Acquisition

**Purpose:** Download all raw data from the NBA API and save to `data/raw/`.

## What this notebook does
1. Sets up directory structure
2. Downloads **team game logs** for each of the 20 seasons (2005-06 → 2024-25) via `TeamGameLogs`
3. Downloads **season-level advanced team stats** via `LeagueDashTeamStats`
4. Downloads the **full league game finder** dataset via `LeagueGameFinder`
5. Saves everything as CSV to `data/raw/`
6. Confirms acquisition with shape & preview output

## ⚠️ Rate Limiting
The NBA API (stats.nba.com) enforces rate limits. This notebook uses `time.sleep(1)` between requests.  
**Do not remove the sleep calls** — you will get throttled and your IP may be temporarily blocked.

**Estimated runtime:** 10–25 minutes for a full fresh download.

## Skip condition
If `data/raw/` already contains all the expected CSV files (e.g., pulled from git), the notebook will detect them and skip re-downloading.

---
## 1. Setup — Directories & Constants

In [ ]:
import os
import time
import pandas as pd
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()           # directory where this notebook lives
RAW_DIR      = NOTEBOOK_DIR / 'data' / 'raw'
GAME_LOGS_DIR  = RAW_DIR / 'game_logs'
TEAM_STATS_DIR = RAW_DIR / 'team_stats'

RAW_DIR.mkdir(parents=True, exist_ok=True)
GAME_LOGS_DIR.mkdir(parents=True, exist_ok=True)
TEAM_STATS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raw data directory  : {RAW_DIR}')
print(f'Game logs directory : {GAME_LOGS_DIR}')
print(f'Team stats directory: {TEAM_STATS_DIR}')

# ── Season list ────────────────────────────────────────────────────────────
# NBA API season format: '2005-06', '2006-07', ..., '2024-25'
def make_season_id(start_year: int) -> str:
    return f'{start_year}-{str(start_year + 1)[-2:]}'

SEASONS = [make_season_id(y) for y in range(2005, 2025)]   # 2005-06 through 2024-25
print(f'\nSeasons to collect ({len(SEASONS)}): {SEASONS[0]} → {SEASONS[-1]}')

# ── Rate-limiting helper ────────────────────────────────────────────────────
SLEEP_BETWEEN_REQUESTS = 1.2  # seconds — be polite to stats.nba.com

---
## 2. Download Team Game Logs (per season)

**Endpoint:** `TeamGameLogs`  
**What it provides:** Every team's per-game box score stats for a given season.  
One row = one team's performance in one game (both teams in a game each get a row).  
Key columns include: `TEAM_ID`, `TEAM_ABBREVIATION`, `GAME_ID`, `GAME_DATE`, `MATCHUP`, `WL`,
`PTS`, `FGM`, `FGA`, `FG_PCT`, `FG3M`, `FG3A`, `FG3_PCT`, `FTM`, `FTA`, `FT_PCT`,
`OREB`, `DREB`, `REB`, `AST`, `TOV`, `STL`, `BLK`, `PLUS_MINUS`.

In [ ]:
from nba_api.stats.endpoints import TeamGameLogs

all_game_logs = []
skipped_seasons = []

for season in SEASONS:
    out_path = GAME_LOGS_DIR / f'game_logs_{season}.csv'

    # Skip if already downloaded
    if out_path.exists():
        print(f'  [SKIP] {season} — file already exists')
        skipped_seasons.append(season)
        continue

    retries = 3
    for attempt in range(retries):
        try:
            logs = TeamGameLogs(
                season_nullable=season,
                season_type_nullable='Regular Season',
            )
            df = logs.get_data_frames()[0]
            df['SEASON'] = season   # add season column for easy filtering later
            df.to_csv(out_path, index=False)
            print(f'  [OK]   {season} — {len(df):,} rows saved to {out_path.name}')
            all_game_logs.append(df)
            break
        except Exception as e:
            if attempt < retries - 1:
                print(f'  [RETRY] {season} attempt {attempt + 1} failed: {e}')
                time.sleep(5)
            else:
                print(f'  [ERR]  {season} — {e}')

    time.sleep(SLEEP_BETWEEN_REQUESTS)

# Load any skipped (already-downloaded) seasons into memory
for season in skipped_seasons:
    df = pd.read_csv(GAME_LOGS_DIR / f'game_logs_{season}.csv')
    all_game_logs.append(df)

print(f'\nLoaded game logs for {len(all_game_logs)} seasons.')

In [ ]:
# Combine all seasons into a single dataframe for a quick sanity check
game_logs_all = pd.concat(all_game_logs, ignore_index=True)
print(f'Combined game logs shape: {game_logs_all.shape}')
print(f'Seasons present         : {sorted(game_logs_all["SEASON"].unique())}')
print(f'Win/Loss distribution:\n{game_logs_all["WL"].value_counts()}')
game_logs_all.head(3)

---
## 3. Download Season-Level Advanced Team Stats (per season)

**Endpoint:** `LeagueDashTeamStats`  
**What it provides:** Aggregated season-level stats per team including advanced metrics like
offensive rating, defensive rating, net rating, pace, true shooting %, etc.  
One row = one team's full-season summary.  
Used for season-level features: current win %, conference rank, season-normalized ratings.

In [ ]:
from nba_api.stats.endpoints import LeagueDashTeamStats

all_team_stats = []
skipped_seasons_ts = []

for season in SEASONS:
    out_path = TEAM_STATS_DIR / f'team_stats_{season}.csv'

    if out_path.exists():
        print(f'  [SKIP] {season} — file already exists')
        skipped_seasons_ts.append(season)
        continue

    try:
        stats = LeagueDashTeamStats(
            season=season,
            season_type_all_star='Regular Season',
            measure_type_detailed_defense='Advanced',   # Advanced = OffRtg, DefRtg, NetRtg, Pace, TS%
        )
        df = stats.get_data_frames()[0]
        df['SEASON'] = season
        df.to_csv(out_path, index=False)
        print(f'  [OK]   {season} — {len(df):,} teams saved to {out_path.name}')
        all_team_stats.append(df)
    except Exception as e:
        print(f'  [ERR]  {season} — {e}')

    time.sleep(SLEEP_BETWEEN_REQUESTS)

for season in skipped_seasons_ts:
    df = pd.read_csv(TEAM_STATS_DIR / f'team_stats_{season}.csv')
    all_team_stats.append(df)

print(f'\nLoaded advanced team stats for {len(all_team_stats)} seasons.')

In [ ]:
team_stats_all = pd.concat(all_team_stats, ignore_index=True)
print(f'Combined advanced team stats shape: {team_stats_all.shape}')
print(f'Columns: {list(team_stats_all.columns)}')
team_stats_all.head(3)

---
## 4. Download League Game Finder (all games, single call)

**Endpoint:** `LeagueGameFinder`  
**What it provides:** Every NBA regular-season game result across all seasons in a single request.
Useful as a flat index of all games: `GAME_ID`, `GAME_DATE`, `TEAM_ID`, `TEAM_ABBREVIATION`,
`MATCHUP`, `WL`, `PTS`, `SEASON_ID`.  
We pull all regular-season games from 2005 onwards.

In [ ]:
from nba_api.stats.endpoints import LeagueGameFinder

lgf_path = RAW_DIR / 'league_game_finder.csv'

if lgf_path.exists():
    print(f'[SKIP] league_game_finder.csv already exists — loading from disk.')
    lgf_df = pd.read_csv(lgf_path)
else:
    print('Downloading league game finder data (season by season)...')
    lgf_frames = []

    for season in SEASONS:
        retries = 3
        for attempt in range(retries):
            try:
                lgf = LeagueGameFinder(
                    season_nullable=season,
                    season_type_nullable='Regular Season',
                    league_id_nullable='00',   # '00' = NBA
                )
                df = lgf.get_data_frames()[0]
                lgf_frames.append(df)
                print(f'  [OK]   {season} — {len(df):,} rows')
                break
            except Exception as e:
                if attempt < retries - 1:
                    print(f'  [RETRY] {season} attempt {attempt + 1} failed: {e}')
                    time.sleep(5)
                else:
                    print(f'  [ERR]  {season} — {e}')

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    lgf_df = pd.concat(lgf_frames, ignore_index=True)

    # SEASON_ID format from this endpoint: '22005', '22006', ..., '22024' (prefix '2' = regular season)
    lgf_df['SEASON_YEAR'] = lgf_df['SEASON_ID'].astype(str).str[1:].astype(int)
    lgf_df = lgf_df[lgf_df['SEASON_YEAR'] >= 2005].copy()

    lgf_df.to_csv(lgf_path, index=False)
    print(f'[OK] Saved {len(lgf_df):,} rows to {lgf_path.name}')

print(f'League game finder shape: {lgf_df.shape}')
lgf_df.head(3)

---
## 5. Acquisition Summary

Confirm all expected files are present in `data/raw/`.

In [ ]:
print('=== Files in data/raw/ ===')
print(f'\n  league_game_finder.csv')

print(f'\n  game_logs/ ({len(list(GAME_LOGS_DIR.glob("*.csv")))} files)')
for f in sorted(GAME_LOGS_DIR.glob('*.csv')):
    print(f'    {f.name}  ({f.stat().st_size / 1_000_000:.2f} MB)')

print(f'\n  team_stats/ ({len(list(TEAM_STATS_DIR.glob("*.csv")))} files)')
for f in sorted(TEAM_STATS_DIR.glob('*.csv')):
    print(f'    {f.name}  ({f.stat().st_size / 1_000_000:.2f} MB)')

# ── Completeness checks ────────────────────────────────────────────────────
expected_game_log_files = {f'game_logs_{s}.csv' for s in SEASONS}
actual_game_log_files   = {f.name for f in GAME_LOGS_DIR.glob('*.csv')}
missing = expected_game_log_files - actual_game_log_files
if missing:
    print(f'\n⚠️  Missing game log files: {sorted(missing)}')
else:
    print(f'\n✅  All {len(SEASONS)} season game log files present.')

expected_ts_files = {f'team_stats_{s}.csv' for s in SEASONS}
actual_ts_files   = {f.name for f in TEAM_STATS_DIR.glob('*.csv')}
missing_ts = expected_ts_files - actual_ts_files
if missing_ts:
    print(f'⚠️  Missing team stats files: {sorted(missing_ts)}')
else:
    print(f'✅  All {len(SEASONS)} season team stats files present.')

if (RAW_DIR / 'league_game_finder.csv').exists():
    print('✅  league_game_finder.csv present.')
else:
    print('⚠️  league_game_finder.csv is MISSING.')

In [ ]:
# Final row count summary
print('=== Row count summary ===')
print(f'  game_logs combined   : {len(game_logs_all):>8,} rows  (target: ~52,000+)')
print(f'  team_stats combined  : {len(team_stats_all):>8,} rows  (30 teams × 20 seasons = ~600)')
print(f'  league_game_finder   : {len(lgf_df):>8,} rows')

print('\n=== game_logs column list ===')
print(list(game_logs_all.columns))

print('\n=== Seasons in game_logs ===')
print(sorted(game_logs_all['SEASON'].unique()))

---
## 6. Next Steps

Data acquisition is complete. Proceed to:

- **`02_eda.ipynb`** (Sejal) — Exploratory data analysis, distributions, and hypothesis testing:
  - Test 1: Home court advantage (one-sample proportion z-test + permutation test)
  - Test 2: Rest day effect (permutation test on win rate difference)

- **`03_cleaning_features.ipynb`** (Jaime) — Cleaning, wrangling, and feature engineering:
  - Rolling 5-game and 10-game averages per team
  - Advanced metric joins
  - Back-to-back flags, rest days, win streak features
  - Opponent strength features
  - Season-normalized features to account for era drift

- **`04_modeling.ipynb`** (Aditya) — Modeling and evaluation:
  - Logistic Regression, Random Forest, XGBoost, Ensemble
  - Time-aware cross-validation (train on earlier seasons, test on later)
  - Metrics: accuracy, AUC-ROC, precision, recall, F1
  - Baseline comparison: home-team-always-wins (~60%)